<a href="https://colab.research.google.com/github/Sarthak-kshirsagar/Ansible-Assignments/blob/main/RAG_Ansible_Storage_Protect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!python3 -m pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [1]:
!pip install ibm-watsonx-ai --upgrade
!pip install faiss-cpu
!pip install sentence-transformers
!pip install ansible


  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nvjitlink_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
Using cached nvidia_cublas_

In [ ]:
import faiss
import json
import yaml
import numpy as np
from sentence_transformers import SentenceTransformer
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
import getpass


encoder = SentenceTransformer("all-MiniLM-L6-v2")

# Load Ansible Knowledge Base JSON
def load_knowledge_base():
    with open("/content/knowledge_base.json", "r") as f:
        return json.load(f)

knowledge_base = load_knowledge_base()

# Create FAISS index for semantic search
def build_faiss_index(knowledge_base):
    descriptions = []
    module_names = []

    for mod_name, details in knowledge_base["Modules"].items():
        descriptions.append(details["description"])
        module_names.append(mod_name)

    embeddings = encoder.encode(descriptions, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)

    return index, module_names

faiss_index, module_names = build_faiss_index(knowledge_base)

# Retrieve best-matching modules
def find_relevant_modules(user_prompt, faiss_index, module_names):
    query_embedding = encoder.encode([user_prompt], convert_to_numpy=True)
    D, I = faiss_index.search(query_embedding, k=2)
    return [module_names[idx] for idx in I[0]]

# Fetch module directives
def get_module_directives(module_name):
    return knowledge_base["Modules"].get(module_name, {}).get("variables", {})

# Authenticate with IBM WatsonX AI
credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Enter your IBM WatsonX API Key: "),
)

# Strict Playbook Generation
def generate_ansible_playbook(user_prompt,model_id):

    matched_modules = find_relevant_modules(user_prompt, faiss_index, module_names)
    print(f"Here are relevant modules found {matched_modules}")

    # Extract directives for each matched module
    modules_data = {}
    for module in matched_modules:
        modules_data[module] = get_module_directives(module)
        print(modules_data)

    print(f"printing the modules data -> {modules_data}")

    # Ensure AI only uses valid directives
    instruction = f"""
    You are a ansible expert.
    I will share a json which contains the ansible modules and roles.
    Now based on the shared json you have to choose the relevant module or role and create a playbook with the variables passed in the json itself.
    This variables are the directives of that module or role.
    Understand the users prompt and then create the playbook and make sure you follow all the best practices while writing the ansible playbook.
    Also the json may contain some roles or modules , so modify the playbook accordingly to include the roles or modules.
    Input:
    Here is the json.
    {json.dumps(modules_data, indent=4)}
    User request: {user_prompt}
    """

    parameters = {
        GenParams.DECODING_METHOD: "greedy",
        GenParams.MAX_NEW_TOKENS: 4000,
        GenParams.STOP_SEQUENCES: ["<end·of·code>"]
    }
# meta-llama/llama-3-1-70b-instruct
# ibm/granite-8b-code-instruct
    model = ModelInference(
        model_id=model_id,
        params=parameters,
        credentials=credentials,
        project_id = "68ff7a4b-b299-4b8f-9211-63185899dee6"
    )

    result = model.generate_text(instruction)
    print("================================")
    print(f"Playbook generated by: {model_id}")
    print("================================")
    return result

user_input = "Write a playbook to install ba client on remote vms"

generated_playbook_granite = generate_ansible_playbook(user_input,"ibm/granite-8b-code-instruct")
print(generated_playbook_granite)
# generated_playbook_llama = generate_ansible_playbook(user_input,"meta-llama/llama-3-1-70b-instruct")
# print(generated_playbook_llama)

